## Image preparation

Remove duplicates from input directory by evaluating the file content hash.

* Note 1: It can also be used to resize images to a defined size (e.g. 32x32 pixels) from an input directory (Input_Dir) into an output directory (Output_Dir).
* Note 2: It's not mandatory to resize with this notebook, because it will be performed in the training notebooks (`02_Train-CNN_...`)

### Basic Parameter

IMPORTANT: Do not rename any variables in this section — they are externally referenced in the GitHub action `Train Model`.

* `Input_Dir`: Path to the input directory containing unscaled training images
* `Output_Dir`: Path to the output directory where resized and scaled images will be saved.
  `Save_Resized_Images`: Set to true to save resized images (Optional: Images will be resized in training notebook automatically)
* `Input_Shape`: Image dimensions (width, height, channels)

In [1]:
# Parameters
Input_Dir = 'data_raw_all'
Output_Dir = 'data_resize_all'

# Save resized images
Save_Resized_Images = False

# Input image size [width, height, channels]
Input_Shape = (32, 32, 3)


In [2]:
# Parameters
Input_Dir = "data_raw_all"
Output_Dir = "data_resize_all"


### Load Libraries

In [3]:
import glob
import os
import sys

from pathlib import Path
import hashlib

import tensorflow as tf


# Prepare folders
if not (Path(Input_Dir).exists() and Path(Input_Dir).is_dir()): # Check if input is availabe
    sys.exit(f"Folder '{Input_Dir}' does not exist.")

if (Save_Resized_Images):
    Path(Output_Dir).mkdir(parents=True, exist_ok=True)  # Create output folder if it doesn't exist


2025-05-28 21:04:24.786453: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-28 21:04:24.789442: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-28 21:04:24.795783: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748466264.810039    2220 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748466264.814217    2220 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-28 21:04:24.829641: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU ins

### Delete Output Directory (Resized Images)

In [4]:
if os.path.exists(Output_Dir):
    files = glob.glob(Output_Dir + '/*.jpg')

    # Delete files
    for f in files:
        os.remove(f)

    print(str(len(files)) + " files have been deleted.")
else:
    print(f"No files have been deleted.")


0 files have been deleted.


### Load Files and Hash Data (optional: Resize Images)

In [5]:
files = glob.glob(Input_Dir + '/*.jpg')
hashes = {}

# Process files
for i, file in enumerate(files):
    # Read image file
    image_bytes = tf.io.read_file(file)
    image = tf.image.decode_image(image_bytes, channels=Input_Shape[2], expand_animations=False)

    # Compute SHA256 hash of image content only, not of file itself (exclude metadata, etc)
    hash_val = hashlib.sha256(tf.io.encode_jpeg(image).numpy()).hexdigest()
    if hash_val in hashes:
        hashes[hash_val].append(file)
        continue
    else:
        hashes[hash_val] = [file]

    # Resize only if activated (only for visualization and debug purpose)
    if (Save_Resized_Images):
        # Process resizing (result: float)
        image = tf.image.resize(image, [Input_Shape[0], Input_Shape[1]], method=tf.image.ResizeMethod.MITCHELLCUBIC)
        # Cast to uint8 (to be close to resizing behaviour using STB library on ESP32)
        image = tf.clip_by_value(image, 0.0, 255.0)
        image = tf.cast(image, tf.uint8)
    
        # Encode as JPEG
        image = tf.io.encode_jpeg(image, quality=100)
    
        # Save resized image
        base = os.path.basename(file)
        save_path = os.path.join(Output_Dir, base)
        tf.io.write_file(save_path, image)

    # Log every 500th image
    if i % 500 == 0:
        print(f" {i} files processed...")

print(f"Completed. {i+1} files processed")


 0 files processed...


2025-05-28 21:04:28.266773: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


 500 files processed...


 1000 files processed...


 1500 files processed...


Completed. 1739 files processed


### Remove Duplicates (from Source Directory)

In [6]:
# Duplicate files are a risk to the metrics, they pollute the validation dataset
for hash in hashes:
    if len(hashes[hash]) > 1:
        print(f"Duplicates: {hashes[hash]}")    
        for duplicate in hashes[hash][1:]:
            # Remove all except the first
            print(f"Removed: {duplicate}")
            os.remove(duplicate)
